In [6]:
import torch
import pandas as pd
import random
import numpy as np
SEED = 39
def set_seed(seed = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()
dataset = pd.read_table("data/dataset_small.tsv", quoting=3, names=["ru_id", "russian", "en_id", "english"])
dataset.head()

,ru_id,russian,en_id,english
0,243,Один раз в жизни я делаю хорошее дело... И оно...,3257,For once in my life I'm doing a good deed... A...
1,5409,Давайте что-нибудь попробуем!,1276,Let's try something.
2,5410,Мне пора идти спать.,1277,I have to go to sleep.
3,5411,Что ты делаешь?,16492,What are you doing?
4,5411,Что ты делаешь?,511884,What do you make?


In [7]:
dataset.shape

(824903, 4)

In [8]:
from src.preprocessing import preprocess_parallel_data
data, _ = preprocess_parallel_data(dataset)

In [9]:
data.shape

(824074, 4)

In [10]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(data, train_size = 0.8, random_state = SEED, shuffle = True)
valid, test = train_test_split(temp, train_size = 0.5, random_state = SEED, shuffle = True)
print(f"train: {train.shape}, valid: {valid.shape}, test: {test.shape}")

train: (659259, 4), valid: (82407, 4), test: (82408, 4)


In [ ]:

import sentencepiece as spm
ru_sp = spm.SentencePieceProcessor(
    model_file="artifacts/tokenizers/ru_bpe.model"
)

en_sp = spm.SentencePieceProcessor(
    model_file="artifacts/tokenizers/en_bpe.model"
)

print(ru_sp.encode("Мне пора идти спать.", out_type=str))
print(en_sp.encode("I have to go to sleep.", out_type=str))

print("RU vocab:", ru_sp.get_piece_size())
print("EN vocab:", en_sp.get_piece_size())

In [12]:
def encode_source(text):
    return ru_sp.encode(text, out_type=int)


def encode_target(text):
    token_ids = en_sp.encode(text, out_type=int)

    return (
        [en_sp.bos_id()] +
        token_ids +
        [en_sp.eos_id()]
    )

In [13]:
for split in [train, valid, test]:
    split["ru_ids"] = split["russian"].apply(encode_source)
    split["en_ids"] = split["english"].apply(encode_target)

In [14]:
RU_PAD_IDX = ru_sp.pad_id()
EN_PAD_IDX = en_sp.pad_id()

RU_UNK_IDX = ru_sp.unk_id()
EN_UNK_IDX = en_sp.unk_id()

EN_SOS_IDX = en_sp.bos_id()
EN_EOS_IDX = en_sp.eos_id()

print(RU_PAD_IDX, EN_PAD_IDX, EN_SOS_IDX, EN_EOS_IDX)

0 0 2 3


In [15]:
row = train.iloc[0]

print(row["russian"])
print(row["ru_ids"])
print(ru_sp.decode(row["ru_ids"]))

print(row["english"])
print(row["en_ids"])
print(en_sp.decode(row["en_ids"][1:-1]))

Вам надо было сразу же мне позвонить.
[473, 321, 218, 2152, 410, 102, 1744, 7804]
Вам надо было сразу же мне позвонить.
You should've phoned me right away.
[2, 86, 271, 7807, 23, 4020, 67, 418, 660, 7799, 3]
You should've phoned me right away.


In [16]:
from torch.utils.data import DataLoader

from src.dataset import (
    TranslationDataset,
    TranslationCollator,
)
BATCH_SIZE = 64
generator = torch.Generator().manual_seed(SEED)
train_dataset = TranslationDataset(train)
val_dataset = TranslationDataset(valid)
test_dataset = TranslationDataset(test)

collator = TranslationCollator(
    ru_pad_idx=RU_PAD_IDX,
    en_pad_idx=EN_PAD_IDX
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    generator=generator
)

valid_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

In [17]:
sources, source_lengths, targets = next(iter(train_loader))

print("Sources:", sources.shape)
print("Lengths:", source_lengths[:10])
print("Targets:", targets.shape)

Sources: torch.Size([64, 26])
Lengths: tensor([ 7, 11,  6,  8, 14,  4,  4,  7, 13,  6])
Targets: torch.Size([64, 27])


In [18]:
import torch.nn as nn
from src.model_baseline import Encoder, Decoder, Seq2Seq
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EMBEDDING_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.2

encoder = Encoder(
    input_dim=ru_sp.get_piece_size(),
    embed_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=RU_PAD_IDX,
    dropout=DROPOUT
)

decoder = Decoder(
    output_dim=en_sp.get_piece_size(),
    embed_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=EN_PAD_IDX,
    dropout=DROPOUT
)

model = Seq2Seq(encoder, decoder).to(DEVICE)
criterion = nn.CrossEntropyLoss(
    ignore_index=EN_PAD_IDX
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [19]:
from src.train import train_epoch, validate_epoch
import copy
import time

NUM_EPOCHS = 10

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE,
        target_pad_idx=EN_PAD_IDX
    )

    val_loss = validate_epoch(
        model,
        valid_loader,
        criterion,
        DEVICE,
        target_pad_idx=EN_PAD_IDX,
        teacher_forcing_rate=0.0
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"time: {epoch_time:.1f}s"
    )

Epoch 01/10 | train loss: 3.0591 | val loss: 3.0697 | time: 511.6s
Epoch 02/10 | train loss: 2.2414 | val loss: 2.7746 | time: 517.6s
Epoch 03/10 | train loss: 1.9920 | val loss: 2.6383 | time: 510.8s
Epoch 04/10 | train loss: 1.8558 | val loss: 2.5634 | time: 504.5s
Epoch 05/10 | train loss: 1.7677 | val loss: 2.5220 | time: 505.8s
Epoch 06/10 | train loss: 1.7034 | val loss: 2.4879 | time: 507.9s
Epoch 07/10 | train loss: 1.6545 | val loss: 2.4602 | time: 518.0s
Epoch 08/10 | train loss: 1.6150 | val loss: 2.4608 | time: 517.6s
Epoch 09/10 | train loss: 1.5790 | val loss: 2.4009 | time: 512.5s
Epoch 10/10 | train loss: 1.5559 | val loss: 2.4104 | time: 511.2s


In [ ]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": best_model_state,
    "model_config": {
        "input_dim": ru_sp.get_piece_size(),
        "output_dim": en_sp.get_piece_size(),
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
    },
    "tokenizer_config": {
        "type": "sentencepiece_bpe",
        "ru_model_path": "artifacts/tokenizers/ru_bpe.model",
        "en_model_path": "artifacts/tokenizers/en_bpe.model",
    },
    "best_val_loss": best_val_loss,
}

torch.save(checkpoint, "models/best_seq2seq_bpe_baseline.pt")

In [21]:
import re
import torch
from tqdm.auto import tqdm

model.load_state_dict(best_model_state)
model.to(DEVICE)
model.eval()

def translate_sentence_bpe(
    sentence,
    model,
    ru_sp,
    en_sp,
    device,
    max_length=50
):
    model.eval()

    source_ids = ru_sp.encode(sentence, out_type=int)

    source = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=device
    )

    source_lengths = torch.tensor(
        [len(source_ids)],
        dtype=torch.long
    )

    predicted_ids = []

    with torch.no_grad():
        _, hidden, cell = model.encoder(source, source_lengths)

        input_token = torch.tensor(
            [en_sp.bos_id()],
            dtype=torch.long,
            device=device
        )

        for _ in range(max_length):
            prediction, hidden, cell = model.decoder(
                input_token,
                hidden,
                cell
            )

            next_token_id = prediction.argmax(dim=1).item()

            if next_token_id == en_sp.eos_id():
                break

            predicted_ids.append(next_token_id)

            input_token = torch.tensor(
                [next_token_id],
                dtype=torch.long,
                device=device
            )

    return en_sp.decode(predicted_ids)

In [22]:
for _, row in valid.sample(5, random_state=SEED).iterrows():
    prediction = translate_sentence_bpe(
        row["russian"],
        model,
        ru_sp,
        en_sp,
        DEVICE
    )

    print("RU:       ", row["russian"])
    print("REFERENCE:", row["english"])
    print("PREDICT:  ", prediction)
    print()

RU:        Не заставляйте меня это есть, пожалуйста.
REFERENCE: Please don't make me eat that.
PREDICT:   Please don't make me eat that.

RU:        Она обещала мне позвонить.
REFERENCE: She promised to give me a phone call.
PREDICT:   She promised me to call me.

RU:        Том сказал, что не хочет с нами идти.
REFERENCE: Tom said that he didn't want to go with us.
PREDICT:   Tom said that he didn't want to go with us.

RU:        Я начинаю уставать от твоих жалоб.
REFERENCE: I'm starting to get tired of your complaining.
PREDICT:   I'm starting to get your complaints.

RU:        Мы хотим увидеть результаты.
REFERENCE: We want to see results.
PREDICT:   We want to see the results.



In [24]:
import sacrebleu

predictions = [
    translate_sentence_bpe(
        russian,
        model,
        ru_sp,
        en_sp,
        DEVICE
    )
    for russian in tqdm(valid["russian"])
]

references = valid["english"].tolist()

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

print(f"Validation BLEU: {bleu.score:.2f}")
print(f"Validation chrF: {chrf.score:.2f}")

  0%|          | 0/82407 [00:00<?, ?it/s]

Validation BLEU: 43.09
Validation chrF: 58.14
